# (first) eda (exploratory data analysis)

data was collected by querying linka's api (airelibre's backend)

in 1 hour windows

for a span of 5+ years, starting at `2020-11-01T00:00:00Z`

(you can go straight to the [summary](#summary))

In [ ]:
import duckdb
import matplotlib.pyplot as plt

In [ ]:
from src.config import CONFIG

DB_PATH = "../" + CONFIG["paths"]["db_path"]
RAW_SCHEMA = 'raw'
RAW_TABLE = RAW_SCHEMA + '.readings'

In [ ]:
con = duckdb.connect(DB_PATH)

## data shape

In [ ]:
con.query(f"DESCRIBE {RAW_TABLE}")

In [ ]:
# let's first unnest the results
df = con.query(f"""
WITH unnested as (
    SELECT
        start,
        "end",
        UNNEST(data) AS item
    FROM {RAW_TABLE}
)
SELECT
    start,
    "end",
    item.*
FROM unnested
""").to_df()

In [ ]:
df.info()

In [ ]:
# we still have a nested object (quality)...
df.iloc[0]["quality"]
# it has 2 components: category and index (the AQI)

In [ ]:
df = con.query(f"""
WITH unnested as (
    SELECT
        start,
        "end",
        UNNEST(data) AS item
    FROM {RAW_TABLE}
)
SELECT
    start,
    "end",
    item.source AS "source",
    item.sensor AS sensor,
    item.description AS description,
    item.latitude
        AS latitude,
    item.longitude
        AS longitude,
    item.quality.category AS quality_category,
    CAST(item.quality.index AS INTEGER) AS quality_index
FROM unnested
""").to_df()

## quantitative values (stats)

In [ ]:
df[['quality_index', 'latitude', 'longitude', 'start', 'end']].describe()

- datetimes: as expected


- (air) quality index (AQI):

    >values range from 0 to 499, so that's reasonable
    >
    >average AQI is 41.6, which is in the (far-end) 'good' range
    >
    >(see [https://www.airnow.gov/aqi/aqi-basics/](https://www.airnow.gov/aqi/aqi-basics/) for details)

- location (latitude, longitude):

    >sensors are [located](https://airelib.re/) within paraguayan territory,
    >the majority being clustered around asuncion-paraguay
    >(coordinates: [(-25.2800, -57.6344)](https://www.openstreetmap.org/?mlat=-25.284&mlon=-59.013#map=7/-25.284/-59.013))
    >
    >the mean value is ...reasonable, outside of asuncion but close enough
    >[-25.1248, -57.7209](https://www.openstreetmap.org/?#map=7/-25.125/-57.721))
    >
    >most values appear to be in that vicinity... BUT max values are positive! better check that
    >
**note:** 4 decimal-digits precision suffices for our purposes (mapping a sensor location to a set of readings)

<details>
<summary>why 4 decimal digits are enough?</summary>

(latitude, longitude) coordinates we are using spherical coordinates: 2 angles (sexagesimal by convention) that let us map every point on (the surface of) a (3D) sphere 


the sphere being the earth, with radius R = 6371 km

its circumference is then C =2*\pi*R ~= 40030 km

that's what 360 degrees of latitude cover: one earth circumference

(starting from the equator: increasing latitude takes you toward the north pole, decreasing to the south pole)

1 latitude degree -> 40030/360 = 111.2 km
=> 0.0001 lat deg -> 11 m

now the distance span by a longitude degree will depend on the latitude we are at: at zero latitude (equator) it's the earth radius, but for other values it is less than that: it's R*cos(lat)

at lat = -25.2800, cos(25.28 deg) = 0.9026

and 1 longitude degree -> 111.2*0.9026 = 100.5 km
=> 0.0091 lon deg -> 10 m (at lat -25.2800)
</details>

# exploration-staging 

let's better save those flatened results, to query them

In [ ]:
con.query(f"""
CREATE OR REPLACE SCHEMA eda;
CREATE OR REPLACE TABLE eda.unnested_raw_readings AS
WITH unnested as (
    SELECT
        start,
        "end",
        UNNEST(data) AS item
    FROM {RAW_TABLE}
)
SELECT
    start,
    "end",
    item.source AS "source",
    item.sensor AS sensor,
    item.description AS description,
    item.latitude
        AS latitude,
    item.longitude
        AS longitude,
    item.quality.category AS quality_category,
    CAST(item.quality.index AS INTEGER) AS quality_index
FROM unnested
""")

## sensor identification

the columns
- `sensor`
- `source`
- `description`

will identify the sensors

In [ ]:
con.query("SELECT DISTINCT sensor FROM eda.unnested_raw_readings")
# this is the sensor type (PM = particulate matter)

In [ ]:
con.query("SELECT DISTINCT source FROM eda.unnested_raw_readings")
# and this would correspond to the (unique?) identifier

In [ ]:
con.query("""
SELECT
    source,
    COUNT(DISTINCT sensor) sensor_type_count
FROM eda.unnested_raw_readings
GROUP BY source
ORDER BY 2 DESC
""")
# it's not unique...
# (only in a few cases there's more than one 'sensor' (type) for a given 'source' value)

In [ ]:
con.query("""
SELECT DISTINCT
    source,
    description,
    sensor
FROM eda.unnested_raw_readings
ORDER BY 1
""")
# descriptions indicate location (also testing purposes, in some cases)
# notice: some descriptions hint at sensors changing location throughout time

## sensor record/readings counts

In [ ]:
counts = con.query("""
SELECT
    source,
    sensor,
    COUNT(*) readings_count,
    ROUND(COUNT(*)/(24*365), 2) readings_count_to_years
FROM eda.unnested_raw_readings
GROUP BY source, sensor
ORDER BY 3 DESC
""")
counts
# (1 reading = 1 hour)


In [ ]:
_ = counts.df().plot.bar(y='readings_count_to_years', xticks=[], legend=None,
    title='readings_count_to_years per (source, sensor) pair', ylabel='years')
# just some ugly plots for now

## reading time-ranges

In [ ]:
q = con.query("""
SELECT
    source,
    sensor,
    COUNT(*) readings_count,
    MIN(start) min_start,
    MAX(d.end) max_end
FROM eda.unnested_raw_readings d
GROUP BY source, sensor
ORDER BY 3 DESC
""")
q

## AQI samples

Let's take a look at some data... from the sensors with the most data


In [ ]:

df = con.query("""
WITH best_sensors AS (
    SELECT
        source,
        sensor,
        COUNT(*) readings_count
    FROM eda.unnested_raw_readings
    GROUP BY source, sensor
    ORDER BY 3 DESC
    LIMIT 3
)
SELECT
    source,
    sensor,
    start dt,
    quality_index aqi
FROM eda.unnested_raw_readings d
JOIN best_sensors USING (source, sensor)
ORDER BY 3
""").df()

In [ ]:
for source, group in df.groupby('source'):
    plt.plot(group['dt'], group['aqi'], alpha=0.5, linewidth=0.2)
plt.xlabel('datetime')
plt.ylabel('AQI')
plt.show()

In [ ]:
sources = df['source'].unique()
fig, axes = plt.subplots(
    len(sources), 1,
    sharex=True, sharey=True)

colors = ['red', 'blue', 'green']
for i, (ax, source) in enumerate(zip(axes, sources, strict=False)):
    data = df[df['source'] == source]
    ax.plot(data['dt'], data['aqi'], color=colors[i], linewidth=0.1)
    #ax.set_title(source)
    ax.set_ylabel('AQI')

axes[-1].set_xlabel('datetime')
plt.tight_layout()
plt.show()

Alright! one can see some little gaps here and there but nice coverage overall

Some of those coordinated high-AQI times may be related to real events. We'll see...

## sensor locations

In [ ]:
locations = con.query("""
SELECT DISTINCT
    source,
    sensor,
    ROUND(latitude, 4) lat,
    ROUND(longitude, 4) lon
FROM eda.unnested_raw_readings
ORDER BY lat, lon, source
""")
locations


intesting. some (source, sensor) pairs appear more than once, with different locations

(that'd be the same sensor either moving to another area
or just updating coordinates)

---

let's try reducing the precision for the above query (0.01 deg -> 1 km)


In [ ]:
locations = con.query("""
SELECT DISTINCT
    source,
    sensor,
    ROUND(latitude, 2) lat,
    ROUND(longitude, 2) lon
FROM eda.unnested_raw_readings
ORDER BY lat, lon, source
""")
locations
# that got rid of some rows

In [ ]:
ldf = locations.df()

ax = ldf.plot.scatter(x='lat', y='lon', c=ldf['source'].astype('category').cat.codes,
    cmap='Set2', colorbar=False, alpha=0.8, title='sensor coordinates')

ax.scatter(-25.28000, -57.63444, marker='x', color='red', zorder=5, label='asuncion')

_ = ax.legend()

In [ ]:
ldf = locations.df()
fldf = ldf[(-30<ldf.lat)&(ldf.lat<-20)&(-65<ldf.lon)&(ldf.lon<-40)] # filtered

ax = fldf.plot.scatter(x='lat', y='lon',
    c=fldf['source'].astype('category').cat.codes,
    cmap='Set2', colorbar=False, alpha=0.8, title='sensor coordinates')

ax.scatter(-25.28000, -57.63444, marker='x', color='red', zorder=5, label='asuncion')

_ = ax.legend()

# yes this could have been a better plot... we'll do that later too


In [ ]:
con.query("""
SELECT DISTINCT
    source,
    sensor,
    description,
    ROUND(latitude, 2) lat,
    ROUND(longitude, 2) lon
FROM eda.unnested_raw_readings
WHERE lat>=0 OR lon>=0
ORDER BY lat, lon, source
""")


those are (source, sensor) pairs with weird locations
most of them describe testing purposes

In [ ]:
con.query("""
SELECT DISTINCT
    source,
    sensor,
    COUNT(DISTINCT ROUND(latitude, 2)) lat_count,
    COUNT(DISTINCT ROUND(longitude, 2)) lon_count
FROM eda.unnested_raw_readings
GROUP BY source, sensor
ORDER BY lat_count DESC, lon_count DESC
""")

confirmed: some sensors move, even when reducing precision significantly (to 2 digits)

that should be taken into account

In [ ]:
con.query("""
SELECT DISTINCT
    source,
    sensor,
    COUNT(DISTINCT ROUND(latitude, 2)) lat_count,
    COUNT(DISTINCT ROUND(longitude, 2)) lon_count
FROM eda.unnested_raw_readings
GROUP BY source, sensor
HAVING lon_count>1 OR lon_count>1
ORDER BY lat_count DESC, lon_count DESC
""")

# summary

this was just a quick exploration of the dataset

by now it is still in a raw state

what we've seen:

- **AQI** (air quality index) values are within the expected range (0 to 500)
- given that **locations** are expected to be within paraguay, some location values are wrong (might be able to clean/fix some, discard the rest)
- sensor **lifespan** is highly variable (some have a very short lifespan and should be better discarded, but others might provide useful readings, even if they are no longer alive)

we didn't check coverage yet (the fraction of the lifespan  with actual data), but we'll get to that


the **most important** thing to consider is the following:

## unit of analysis: `located_sensor`

while the fields `source` (hash identifier, in most cases), `sensor` (device type) and `description` (name) allow us do identify a measuring device,

what we care about is the AQI readings at a given location: the more sensors we have in an area, the better we can assess the air quality there (even if some malfunction)


so, a sensor that moves to a different location should be then taken as a different entity;

in other words, the unit of analysis will be a `located_sensor`, not just a device: same hardware at different locations will be treated as a distinct source

> `located_sensor` = (
>>
>>    `(source, sensor)` (sensor_hash, sensor_type), 
>>
>>    `(latitude, longitude)` (location, with 4 decimal-digits precision for each coordinate)
>
>)

besides that, a description might change (we better take the latest value)

and a location might change by a few meters and correspond to the same `located_sensor` (so, we'll allow some jitter)